# LOG05 | 미션 10 | Solution — 교관 전용

**학생에게 배포하지 마세요.** 이 Notebook은 근거 확인형 교관 해설, 모의 기록의 검토 예시와 평가 기준을 담습니다. 공개된 원자료가 없는 항목은 판정 불가로 남깁니다.

**미션:** 무인 보급장비 투입 전 상태·책임·중지조건 표준 체크리스트 완성

## 📦 실습자료 업로드 (모든 미션 공통 · 실습 전 필수)

아래 **「LOG05 실습자료 ZIP 업로드 및 압축 해제」** 코드셀을 **먼저 실행**하세요.

1. 교육기관에서 제공한 **`LOG05_실습프로젝트.zip`**을 준비합니다. 파일명이 다르더라도 LOG05 자료가 담긴 정상적인 ZIP 파일이면 업로드할 수 있습니다.
2. **Google Colab:** 셀 실행 후 나타나는 업로드 창에서 ZIP을 선택하면 자동으로 압축을 해제합니다. **로컬 Jupyter:** ZIP 선택 후 **「선택한 ZIP 압축 해제」** 버튼을 누릅니다.
3. 화면에서 **「압축 해제 완료」** 메시지와 실습자료 폴더 경로를 확인한 다음, 아래 미션 실습을 진행하세요.

**주의:** 업로드가 끝나지 않았다면 미션 실습을 시작하지 마세요. ZIP을 올렸다는 사실만으로 실습 결과나 교관 승인이 완료되는 것은 아닙니다. 아래 코드셀의 소스코드는 기본 숨김으로 설정되어 있습니다.


In [ ]:
# @title 📦 LOG05 실습자료 ZIP 업로드 및 압축 해제 { display-mode: "form" }
# 학습자 화면에는 결과와 업로드 인터페이스만 표시하도록 구성합니다.
from pathlib import Path
import os, zipfile, stat
from IPython.display import display, Markdown

LOG05_PROJECT_DIR = Path.cwd() / "LOG05_실습자료"
LOG05_PROJECT_READY = False

def _log05_extract_zip(zip_path):
    global LOG05_PROJECT_READY
    zip_path = Path(zip_path)
    if not zipfile.is_zipfile(zip_path):
        raise ValueError("유효한 ZIP 압축파일이 아닙니다. 실습자료 ZIP을 다시 선택해 주세요.")
    dst = LOG05_PROJECT_DIR.resolve()
    dst.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as archive:
        entries = archive.infolist()
        if len(entries) > 20000 or sum(info.file_size for info in entries) > 1024**3:
            raise ValueError("압축파일의 항목 수 또는 해제 후 용량이 허용 범위를 초과합니다.")
        for info in entries:
            item = Path(info.filename.replace('\\', '/'))
            if item.is_absolute() or '..' in item.parts or not info.filename.strip():
                raise ValueError("안전하지 않은 ZIP 내부 경로가 포함되어 있습니다: " + info.filename)
            target = (dst / item).resolve()
            if not target.is_relative_to(dst):
                raise ValueError("실습 폴더 밖으로 벗어나는 파일 경로입니다.")
            mode = (info.external_attr >> 16) & 0o170000
            if mode == stat.S_IFLNK:
                raise ValueError("심볼릭 링크가 포함된 ZIP은 허용하지 않습니다.")
        for info in entries:
            target = dst / Path(info.filename.replace('\\', '/'))
            if info.is_dir():
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(info) as src, target.open('wb') as out:
                    import shutil
                    shutil.copyfileobj(src, out)
    files = [p for p in dst.rglob('*') if p.is_file()]
    if not files:
        raise ValueError("ZIP에 압축 해제할 실습자료 파일이 없습니다.")
    LOG05_PROJECT_READY = True
    print("✅ LOG05 실습자료 압축 해제 완료:", dst)
    print("📄 확인된 파일 수:", len(files))
    print("ℹ️ 실습자료 폴더:", dst)
    print("※ 업로드·압축 해제는 학습 수행 및 평가 완료를 뜻하지 않습니다.")
    return dst

display(Markdown("**실습 시작 전 필수:** 교육기관에서 제공한 `LOG05_실습프로젝트.zip`을 업로드하세요. 이름이 다르더라도 정상적인 `.zip` 파일을 선택할 수 있지만, **LOG05 실습자료가 포함된 ZIP**이어야 합니다. 업로드 및 압축 해제를 완료한 후 다음 실습 단계로 진행하세요."))
try:
    from google.colab import files as _colab_files
    print("📤 업로드 창에서 LOG05 실습자료 ZIP 파일을 선택하세요.")
    _uploaded = _colab_files.upload()
    _zip_files = [(name, content) for name, content in _uploaded.items() if name.lower().endswith('.zip')]
    if not _zip_files:
        print("⚠️ ZIP 파일이 업로드되지 않았습니다. 이 셀을 다시 실행하세요.")
    else:
        if len(_zip_files) > 1:
            print("ℹ️ 여러 ZIP 중 첫 번째 파일을 사용합니다:", _zip_files[0][0])
        _name, _data = _zip_files[0]
        _zip_path = Path.cwd() / Path(_name).name
        _zip_path.write_bytes(_data)
        _log05_extract_zip(_zip_path)
except ImportError:
    import ipywidgets as _widgets
    _file_input = _widgets.FileUpload(accept='.zip', multiple=False, description='ZIP 파일 선택')
    _extract_button = _widgets.Button(description='선택한 ZIP 압축 해제', button_style='success')
    _upload_message = _widgets.Output()
    def _get_uploaded_zip():
        value = _file_input.value
        if not value:
            raise ValueError('먼저 LOG05 실습자료 ZIP을 선택하세요.')
        item = next(iter(value.values())) if isinstance(value, dict) else value[0]
        name = item['name']
        data = item['content']
        if not name.lower().endswith('.zip'):
            raise ValueError('ZIP 파일만 선택할 수 있습니다.')
        return name, bytes(data)
    def _on_extract(_):
        with _upload_message:
            _upload_message.clear_output()
            try:
                name, data = _get_uploaded_zip()
                target = Path.cwd() / Path(name).name
                target.write_bytes(data)
                _log05_extract_zip(target)
            except Exception as exc:
                print('⚠️ 업로드 또는 압축 해제 실패:', exc)
    _extract_button.on_click(_on_extract)
    display(_widgets.VBox([_file_input, _extract_button, _upload_message]))
    print('로컬 Jupyter: ZIP을 선택한 뒤 「선택한 ZIP 압축 해제」 버튼을 누르세요.')


## 교관 해설 및 판단 예시

최종 8개 장과 부록·문서체계를 확인하고, 가상 자료·AI의 해석·본인 판단·실제 사람 검토를 각각 표시합니다. ZIP 작성만으로 PASS나 공식 승인이 되지 않습니다.

**예시 풀이의 기본 논리**

1. 원본 가상 자료에서 해당 사례·기록 번호, 기간과 분모를 찾습니다.
2. 관찰된 **사실**, AI가 제안한 **해석**, 학생이 내리는 **판단**, 부족한 **자료**를 별도로 적습니다.
3. 근거를 확인할 수 없는 주장에는 승인이나 성공 판정을 붙이지 않습니다.
4. 최초 답안과 수정 답안을 구분하며 AI 요청·답변 원문을 확인합니다.
5. 학생의 독립 수행과 실제 사람 평가 조건을 별도로 확인합니다.

**실제 제3자 평가/승인:** 제공된 문서만으로 자동 생성할 수 없습니다.

## 원자료에 근거한 교관용 작성 예시

### 예시 결과: 최종 패키지 8개 장 구성

1. 교육 목적·가상 자료 범위
2. 원천 자료 및 기록의 정의
3. 교육용 점검표의 구성과 용어
4. 근거 신뢰도와 미확인 상태
5. 문서상 검토·보류 기록
6. 담당 역할 및 검토 책임 표기
7. 가상 사례 분석과 사후 검토
8. 검증 결과·한계·다음 개선 사항

**필수 부록 예시:** 자료 출처표, 미션별 빈 작성 양식, AI 요청·답변 원문, 수정 이력, 실제 수행/모의/미실시 구분, 교관 판정란.

**교관 검토:** 각 장과 부록이 미션 결과와 연결되는지 확인합니다. 사람의 검토가 없는데 승인이라고 기록하거나 실제 장비에 적용 가능한 규정이라고 표시하면 보완 필요입니다.

## 학생과 동일한 원본 자료

미션 10. 무인 보급장비 투입 전 상태·책임·중지조건 표준 체크리스트 완성

**가. 제공 자료 1 - 지침서 8개 장 구성과 가져올 결과물**

| 장 | 제목 | 포함 내용 | 가져올 결과물 |
| --- | --- | --- | --- |
| 1 | 목적과 적용 범위 | 목적, 적용 장비와 상황, 적용하지 않는 상황, 교전수칙과의 관계 | M1 |
| 2 | 용어 정의 | 계속, 감속, 제자리 안전 정지, 복귀, 원격 회수 요청, 임무 취소 | M1·M3 |
| 3 | 운용 환경과 위험 요인 | 구간 특성, 6개 위험 분류, 사례 | M1 |
| 4 | 투입 전 점검 | 점검 항목표, 신뢰도 G1～G6, 교차 확인 규칙 | M2·M9 |
| 5 | 중지·복귀 조건 | 상태 변화별 처리 단계, 정보 공백, 판단 한계 | M3·M7·M8 |
| 6 | 책임과 승인 | 직책별 책임 한계, 승인 분기점, 사후 감사 | M4·M5 |
| 7 | 관제·훈련·평가 | 관제 상황판, 긴급 중지 명령, 스트레스 훈련, 루브릭, AAR | M6～M8 |
| 8 | 개정 관리 | 연간 개정 주기, 의견 수렴 창구, 책임 직책, 이력 | M8·M9 |

**가. 제공 자료 2 - 모의 심의 역할 카드**

| 역할 | 검토 질문 |
| --- | --- |
| 교리 역할 | 점검 항목과 처리 단계가 일관되는가? 신병도 이해할 용어인가? |
| 법무 역할 | '승인과 중지 결정은 사람이, 교전수칙·지휘계통에 따라'가 드러나는가? 장비 파괴·무력화 쟁점이 검토 연계로 처리되는가? |
| 작전 역할 | 현장에서 20분 안에 점검할 수 있는가? 점검 때문에 긴급 보급이 지나치게 늦어지지 않는가? |

**나. 작성 서식 1 - 근거 표기 점검표(지침서 조항마다 작성)**

| 조항 | 기준 내용 요약 | 근거(미션·가상 자료) | 표기 여부 |
| --- | --- | --- | --- |
| 1-1 |  |  |  |
| 2-1 |  |  |  |
| 3-1 |  |  |  |
| 4-1 |  |  |  |
| 5-1 |  |  |  |
| 6-1 |  |  |  |
| 7-1 |  |  |  |
| 8-1 |  |  |  |

**나. 작성 서식 2 - 모의 심의 지적 사항 반영표**

| 번호 | 역할 | 지적 사항 | 반영 내용 | 반영 위치(장·조) |
| --- | --- | --- | --- | --- |
| 1 |  |  |  |  |
| 2 |  |  |  |  |
| 3 |  |  |  |  |
| 4 |  |  |  |  |
| 5 |  |  |  |  |
| 6 |  |  |  |  |

**나. 작성 서식 3 - 개정 관리: 개정 주기 (   ), 의견 수렴 창구와 방법 (   ), 개정 책임 직책(가상) (   ), 개정 이력 기록 위치 (   )**

**다. 점검표**

| 점검 항목 | 확인 |
| --- | --- |
| 8개 장과 부록을 모두 완성했다 | (   ) |
| 모든 기준 조항에 근거(미션·가상 자료)를 표기했다 | (   ) |
| 교리·법무·작전 역할의 지적 사항을 반영하고 반영 위치를 적었다 | (   ) |
| '승인과 중지 결정은 사람이 하며 실제 교전 판단은 교전수칙·지휘계통에 따른다'는 문구가 1장과 6장에 있다 | (   ) |
| 장비 파괴·무력화 조치를 정의하지 않고 검토 연계로 처리한다는 문구가 5장에 있다 | (   ) |
| 연간 개정 주기와 실무 의견 수렴 창구를 정했다 | (   ) |
| 실제 무인체계 제원·성능을 쓰지 않았다 | (   ) |

## 활동별 교관 검토 해설

### 활동 1. 8장·부록의 최종 버전을 모은다

**교관이 확인할 구체적인 수행:** 1. 단계 개요 목표: 미션 1부터 미션 9까지 수행하는 동안 도출된 모든 결과물, 미확인 영역, 실제/모의/미실시 상태의 데이터를 종합하여, 최종 무인 보급장비 표준 지침서 문서 내 8개 핵심 장(Chapter)과 부록 에 정확하게 연결·편성합니다. 수행 방식: 단편적으로 흩어져 있는 개별 실습 결과들을 하나의 통일된 구조로 집대성하되, 단순한 계획 단계를 넘어 실제 수행된 기록과 대조하여 누락이나 허위 입력이 없도록 엄정하게 정돈합니다. 2. 세부 수행 절차 미션 1～9 산출물 및 상태 데이터 통합: 각 미션별로 생성된 평가 루브릭, AAR 피드백, 7건의 재검증 결과, 1페이지 브리핑 등의 최종 버전을 수집합니다. 원본 8장 및 부록 체계에 연결: 수집된 데이터를 표준 운영 매뉴얼( manual/log05_standard_operational_manual.md )과 양식 파일( LOG05_M10_forms.md , forms/LOG05_M10_forms.md )의 지정된 8개 장과 부록 항목에 각각 매핑합니다. 미확인 및 미실시 영역 명시: 실습 과정에서 정보 공백이나 제약으로 인해 완료되지 못한 미실시 항목이나 미확인 영역을 임의로 채우지 않고, 원본 상태 그대로 정확히 반영합니다. 3. 주의 및 검증 사항 계획 단계에 머물렀던 내용을 실제 완료된 것으로 허위 표기하지 않도록 주의해야 합니다. 본 단계에서 편성화된 8장과 부록의 초안은 후속 2단계(AI 및 조항별 근거 검토)의 직접적인 검증 대상이 되므로 데이터의 정합성과 출처를 철저히 대조해야 합니다.

**예시 답안의 근거:** 학생이 원본 자료 번호와 해당 원문·분모·조건을 연결했는지 확인합니다. 입력 자료가 없는 값은 '확인 필요'로 남겨야 합니다.

**평가 포인트:** 이 활동의 확인 근거 M10 활동1 — 8장·부록의 최종 버전을 모은다: M1～M9 결과·미확인·실제/모의/미실시 상태를 원본 8장 에 연결한다. 확인 증거는 실제 제공 자료번호·본인 작성/질문·AI 회신·대조/수정의 해당 위치입니다. 계획만 작성했다면 실행 완료로 표시하지 않습니다. 단계 결과를 forms/LOG05_M10_forms.md와 최종 파일 manual/log05_standard_operational_manual.md, LOG05_M10_forms.md, forms/LOG05_M10_forms.md에 연결합니다. 보완 방법 빠진 입력·파일·근거·규칙을 확인하고 관련 문서/코드를 수정해 같은 조건에서 다시 검증합니다. 실제 수행 기록 없이 실행·승인 완료를 만들어 넣지 않습니다.

### 활동 2. AI와 모든 조항의 근거를 검토한다

**교관이 확인할 구체적인 수행:** 1. 단계 개요 목표: 표준 체크리스트 및 지침서에 포함된 모든 조항을 대상으로 숫자, G 방향(이동 방향 및 방위각), 조건, 5분 기한, 위임 권한, 명령 상태 등의 핵심 변수들이 상호 모순 없이 정확하게 일치하는지 AI와의 대조를 통해 조항별로 엄밀하게 검토합니다. 수행 방식: 단편적인 육안 검토를 넘어, AI와의 질의응답 및 논리 대조 과정을 통해 각 조항에 명시된 수치와 운용 규칙이 실제 현장 지침 및 이전 미션(M1～M9)의 근거 자료와 완벽하게 부합하는지 교차 검증합니다. 2. 세부 수행 절차 핵심 변수 및 조건 조항별 대조 검증: 숫자 및 기준값: 중지 조건, 장비 적재 중량, 허용 오차 등의 수치가 모든 장(Chapter)에 걸쳐 일관되게 반영되었는지 확인합니다. G 방향 및 방위각: 경로 이탈 및 비상 복귀 시 설정된 이동 방향(G 방향)의 정합성을 검토합니다. 시간 제한(5분 기한) 및 위임 권한: 비상 조치 기한(예: 5분 이내 대응)과 현장 조종수 및 군수참모 간 위임 범위가 명확히 규정되었는지 대조합니다. 명령 상태: 하달된 명령과 모의 회신 로그 간의 상태 정의가 상충되지 않는지 점검합니다. AI 활용 근거 검토 및 수정: AI 레드팀 대화 및 검토 프롬프트를 활용하여 각 조항의 논리적 허점을 찾아내고, 지적된 항목을 즉각적으로 수정 보완합니다. 3. 주의 및 검증 사항 형식적인 검토에 그치지 않고, 조항 간 수치나 조건이 충돌하는 부분이 없는지 원본 데이터와 철저히 대조해야 합니다. 본 단계의 검증 결과 및 수정 내역은 양식 파일( forms/LOG05_M10_forms.md , LOG05_M10_forms.md )과 최종 매뉴얼( manual/log05_standar

**예시 답안의 근거:** 학생이 원본 자료 번호와 해당 원문·분모·조건을 연결했는지 확인합니다. 입력 자료가 없는 값은 '확인 필요'로 남겨야 합니다.

**평가 포인트:** 이 활동의 확인 근거 M10 활동2 — AI와 모든 조항의 근거를 검토한다: 숫자·G 방향·조건·5분 기·위임·명령 상태가 일치하는지 조항별로 확인한다. 확인 증거는 실제 제공 자료번호·본인 작성/질문·AI 회신·대조/수정의 해당 위치입니다. 계획만 작성했다면 실행 완료로 표시하지 않습니다. 단계 결과를 forms/LOG05_M10_forms.md와 최종 파일 manual/log05_standard_operational_manual.md, LOG05_M10_forms.md, forms/LOG05_M10_forms.md에 연결합니다. 보완 방법 빠진 입력·파일·근거·규칙을 확인하고 관련 문서/코드를 수정해 같은 조건에서 다시 검증합니다. 실제 수행 기록 없이 실행·승인 완료를 만들어 넣지 않습니다.

### 활동 3. 모의 심의에 직접 대응한다

**교관이 확인할 구체적인 수행:** 1. 단계 개요 목표: 완성된 표준 체크리스트와 지침서 초안을 바탕으로 교리적 타당성, 법무적 적법성, 그리고 작전적 실효성 관점에서 진행되는 가상의 모의 심의(Simulated Review)에 직접 대응하고, 각 심의관들의 지적 사항을 원본 자료와 철저히 대조하여 반영, 보류, 또는 위치 기록 을 명확히 남깁니다. 수행 방식: 형식적인 검토나 일방적인 수용을 지양하고, 실제 군사 작전 및 규정 심의 절차와 동일하게 각 영역별 지적 사항에 대한 방어 논리와 근거를 구축하며, 단순 '승인' 상태와 실제 검토·보완 과정을 엄정하게 구분하여 수행합니다. 2. 세부 수행 절차 교리·법무·작전 영역별 지적 사항 대조 분석: 교리적 심의 대응: 기존 작전 교리와의 정합성을 검토하고, 무인 보급장비 운용 기준이 상위 규정에 저촉되지 않는지 확인합니다. 법무적 심의 대응: 비상 시 자폭/복귀 조건, 책임 한계(군수참모 vs 원격 조종수), 권한 위임 범위 등에 대한 법적 책임과 리스크 요소를 점검합니다. 작전적 심의 대응: 실전 상황에서 부대 작전에 지장을 주지 않고 신속하게 실행 가능한지 운용 타당성을 검증합니다. 심의 결과 조치 결정 (반영·보류·위치 명시): 반영: 지적된 내용이 타당하여 표준 매뉴얼이나 양식 문서의 구체적인 위치( manual/log05_standard_operational_manual.md 등)에 즉시 수정·적용해야 하는 항목. 보류: 추가적인 데이터나 상급 부대 지침 확인이 필요하여 당장 반영하기 어렵고 사유를 남겨야 하는 항목. 위치 기록: 지적 사항과 조치 결과가 매뉴얼 상 어느 장(Chapter) 또는 부록에 반영되었는지 추적 가능하도록 좌표를 명시. 실제 승인과 검토 과정의 엄

**예시 답안의 근거:** 학생이 원본 자료 번호와 해당 원문·분모·조건을 연결했는지 확인합니다. 입력 자료가 없는 값은 '확인 필요'로 남겨야 합니다.

**평가 포인트:** 이 활동의 확인 근거 M10 활동3 — 모의 심의에 직접 대응한다: 교리·법무·작전 지적을 자료로 대조해 반영·보류·위치를 남긴다. 실제 승인과 구분한다. 확인 증거는 실제 제공 자료번호·본인 작성/질문·AI 회신·대조/수정의 해당 위치입니다. 계획만 작성했다면 실행 완료로 표시하지 않습니다. 단계 결과를 forms/LOG05_M10_forms.md와 최종 파일 manual/log05_standard_operational_manual.md, LOG05_M10_forms.md, forms/LOG05_M10_forms.md에 연결합니다. 보완 방법 빠진 입력·파일·근거·규칙을 확인하고 관련 문서/코드를 수정해 같은 조건에서 다시 검증합니다. 실제 수행 기록 없이 실행·승인 완료를 만들어 넣지 않습니다.

### 활동 4. 최종본·개정 상태를 제출한다

**교관이 확인할 구체적인 수행:** 1. 단계 개요 목표: 앞선 1～3단계를 거쳐 완성된 8개 핵심 장(Chapter)과 부록, 필수 준수 문구, 정기 개정 주기(연 1회), 실무자 의견 수렴 창구, 그리고 실제 검토 대기 상태의 데이터를 최종 점검하여 완벽한 형태의 표준 매뉴얼 및 양식 파일로 확정하고 제출합니다. 수행 방식: 형식적인 저장이나 미완성 상태의 제출을 방지하기 위해 필수 행정 및 운영 요건이 모두 포함되었는지 최종적으로 교차 검증하고, 공식적인 제출 상태로 전환합니다. 2. 세부 수행 절차 필수 구성 요소 및 준수 문구 최종 편성화: 8장 및 부록: 미션 1부터 미션 9까지의 모든 분석, 검증, 심의 조치 결과가 반영된 최종 매뉴얼 내용( manual/log05_standard_operational_manual.md ) 정돈. 필수 문구 및 개정 체계: 매뉴얼 내에 규정된 필수 지침 문구와 연 1회 정기 개정 원칙을 명시. 의견 수렴 창구: 현장 운용 중 발생하는 피드백을 지속적으로 수집하기 위한 의견 창구 및 담당 부서 지정. 실제 검토 대기 상태 전환 및 파일 저장: 작성된 최종 표준 매뉴얼과 양식 파일( LOG05_M10_forms.md , forms/LOG05_M10_forms.md )의 무결성을 확인하고, 검수 및 실무 적용을 위한 '검토 대기' 상태로 저장·제출합니다. 3. 주의 및 검증 사항 실제 검토 기록이나 필수 행정 문구(연 1회 개정, 의견 창구 등)가 누락된 채 임의로 완료 처리되지 않도록 세부 항목을 빠짐없이 대조해야 합니다. 본 단계를 끝으로 무인 보급장비 표준 체크리스트 완성 프로젝트의 모든 편성이 최종 마무리됩니다.

**예시 답안의 근거:** 학생이 원본 자료 번호와 해당 원문·분모·조건을 연결했는지 확인합니다. 입력 자료가 없는 값은 '확인 필요'로 남겨야 합니다.

**평가 포인트:** 이 활동의 확인 근거 M10 활동4 — 최종본·개정 상태를 제출한다: 8장·부록·필수 문구·연 1회 개정·의견 창구·실제 검토 대기를 저장한다. 확인 증거는 실제 제공 자료번호·본인 작성/질문·AI 회신·대조/수정의 해당 위치입니다. 계획만 작성했다면 실행 완료로 표시하지 않습니다. 단계 결과를 forms/LOG05_M10_forms.md와 최종 파일 manual/log05_standard_operational_manual.md, LOG05_M10_forms.md, forms/LOG05_M10_forms.md에 연결합니다. 보완 방법 빠진 입력·파일·근거·규칙을 확인하고 관련 문서/코드를 수정해 같은 조건에서 다시 검증합니다. 실제 수행 기록 없이 실행·승인 완료를 만들어 넣지 않습니다.

## 원본 CMS 평가 기준

**평가 항목 1:** [정량적 평가] [필수] 표준 지침서 통합 완성 — 표준 지침서 총 8개 장 및 부록을 100% 완성하고 모든 기준 조항에 근거(미션·가상 자료) 출처를 표기했는가, 모의 심의 지적 사항을 반영하고 연간 개정 주기와 실무 의견 수렴 창구를 두었는가, '승인과 중지 결정은 사람이 하며 실제 교전 판단은 교전수칙·지휘계통에 따른다'는 문구를 명시했는가. PASS 증거 제공된조건·단위·기간·분모와실제수치/건수/항목근거를연결합니다. 목표미달·0기준·미측정은그대로표시합니다. 보완 방법 기준별빠진항목과미달원인을찾아수정하고재검증합니다.

**평가 항목 2:** [정성적 평가] Why/How/What: 본인 계획·실제 대화·근거 검토3건 이상·보완·재확인·최종 반영. PASS 증거 미션 1～9의 산출물을 통합해 무인 보급장비 투입 전 상태·책임·중지조건 표준 체크리스트(표준 지침서)를 완성하고 모의 심의를 거친다.의판단이제공자료·역할책임·교육용가정과연결됩니다. 사실/제안/미확인과실제/모의수행을구분합니다. 보완 방법 근거없는원인·규정·완료추정은수정하거나확인필요로보류합니다.

**평가 항목 3:** [진행 추적 및 완료 기준] 실제 조원·두 평가자·2분 판단·20분 문서점검·3분 발표·실제 교관 확인을 AI 모의와 구분한다. 미실시를 PASS로 쓰지 않는다. 파일: manual/log05_standard_operational_manual.md, LOG05_M10_forms.md, forms/LOG05_M10_forms.md 실제AI요청·답변·근거대조3건이상·보완·재확인·본인최종선택과작성/실행기록이연결됩니다. 최종ZIP생성은실제학습완료·교관승인 인증이아닙니다. 보완 방법 미작성·미실행·검토누락을보완하고교관검토상태를별도로표시합니다.

**교관 검토 기록:** 판정 [미확인 / 보완 필요 / PASS / FAIL]을 실제 기준에 따라 기입해야 하며, 자동 표시는 하지 않습니다.

In [ ]:
MISSION_JSON = '{"code": "LOG05_m10", "steps": [{"title": "8장·부록의 최종 버전을 모은다"}, {"title": "AI와 모든 조항의 근거를 검토한다"}, {"title": "모의 심의에 직접 대응한다"}, {"title": "최종본·개정 상태를 제출한다"}]}'
# LOG05 학습 기록 도구 (표준 Python/Jupyter 환경용)
import json, zipfile, io, os, datetime
from pathlib import Path
from IPython.display import display, Markdown
try:
    import ipywidgets as widgets
except ImportError as exc:
    raise RuntimeError("ipywidgets가 필요합니다. 폐쇄망에서는 사전 설치된 로컬 Jupyter 또는 제공된 Markdown 양식을 사용하세요.") from exc

MISSION_DATA = json.loads(MISSION_JSON)
CODE = MISSION_DATA['code']
OUT_DIR = Path(CODE + '_results')
OUT_DIR.mkdir(exist_ok=True)
BACKUP = OUT_DIR / (CODE + '_record.json')
fields = {}
for no, item in enumerate(MISSION_DATA['steps'], 1):
    fields['activity_%02d' % no] = widgets.Textarea(
        value='', placeholder='내 판단, 확인한 자료 번호, AI 제안과 수정 이유를 직접 작성하세요.',
        description='활동 %s' % no, layout=widgets.Layout(width='98%',height='130px'),
        style={'description_width':'75px'})
for key, label in [('first_thought','AI에게 묻기 전 내 판단'), ('ai_request','실제로 보낸 AI 요청 전체'),
                   ('ai_response','실제로 받은 AI 답변 전체'), ('source_check','원자료 대조 3건과 수정 이유'),
                   ('final_result','최종 작성 결과와 근거'), ('limits','미실시/미측정/확인 필요 및 교관 검토 상태')]:
    fields[key] = widgets.Textarea(value='', description=label,
       layout=widgets.Layout(width='98%',height='125px'),style={'description_width':'190px'})
status = widgets.HTML(value='<b>아직 저장하지 않았습니다.</b> 작성과 교관 승인은 별개입니다.')
restore_input = widgets.Textarea(description='복원 JSON',placeholder='백업 JSON의 내용을 붙여넣으세요.',layout=widgets.Layout(width='98%',height='100px'),style={'description_width':'110px'})

def payload():
    return {'schema':'log05-student-v1','mission':CODE,'saved_at':datetime.datetime.now(datetime.timezone.utc).isoformat(),
            'answers':{key:w.value for key,w in fields.items()},
            'status':'작성 기록 (학습 PASS 또는 교관 승인 아님)'}

def save_record(_=None):
    obj=payload()
    if BACKUP.exists():
        stamp=datetime.datetime.now().strftime('%Y%m%d_%H%M%S_%f')
        BACKUP.rename(OUT_DIR / (CODE+'_previous_'+stamp+'.json'))
    BACKUP.write_text(json.dumps(obj,ensure_ascii=False,indent=2),encoding='utf8')
    status.value='<b>로컬 저장 완료:</b> '+str(BACKUP)+' (PASS 아님)'

def export_json(_=None):
    save_record()
    raw=BACKUP.read_bytes()
    try:
        from google.colab import files
        files.download(str(BACKUP))
    except ImportError:
        print('Colab이 아닙니다. 저장 위치:',BACKUP.resolve())

def restore_record(_=None):
    try:
        obj=json.loads(restore_input.value)
        if obj.get('mission')!=CODE or not isinstance(obj.get('answers'),dict):raise ValueError('다른 미션 또는 형식이 맞지 않는 백업입니다.')
        save_record()  # 복원 전 백업 저장
        for key,w in fields.items():w.value=str(obj['answers'].get(key,''))
        status.value='<b>복원 적용:</b> 기존 작성 내용은 이전 JSON으로 보관했습니다. 다시 저장하세요.'
    except Exception as exc:status.value='<b>복원 오류:</b> '+str(exc)

def export_review_zip(_=None):
    save_record()
    target=OUT_DIR / (CODE+'_review.zip')
    with zipfile.ZipFile(target,'w',compression=zipfile.ZIP_DEFLATED) as zz:
        zz.write(BACKUP,arcname=BACKUP.name)
        text='\n\n'.join('## '+key+'\n'+val for key,val in payload()['answers'].items())
        zz.writestr(CODE+'_work.md',text)
        zz.writestr('README.txt','검토용 보관 파일. 실제 평가 PASS 또는 교관 승인 자료가 아닙니다.')
    try:
        from google.colab import files
        files.download(str(target))
    except ImportError:
        print('Colab이 아닙니다. ZIP 위치:',target.resolve())

for i,item in enumerate(MISSION_DATA['steps'],1):
    display(Markdown('### 활동 %d. %s' % (i,item['title'])))
    display(fields['activity_%02d' % i])
for key,label in [('first_thought','최초 판단'),('ai_request','AI 요청 원문'),('ai_response','AI 답변 원문'),
 ('source_check','원자료 대조'),('final_result','최종 결과'),('limits','한계 및 실제 검토 상태')]:
    display(Markdown('### '+label));display(fields[key])
buttons=[]
for title,handler in [('작성 내용 저장',save_record),('작성 JSON 내려받기',export_json),('검토용 ZIP 내려받기',export_review_zip),('작성 JSON 복원',restore_record)]:
    b=widgets.Button(description=title,button_style='info');b.on_click(handler);buttons.append(b)
display(widgets.HBox(buttons[:2]));display(widgets.HBox(buttons[2:]));display(restore_input);display(status)
